# NSynth + AFTER Representation Evaluation: Informativeness

Clean notebook for training a Synesis linear probe directly from the NSynth/AFTER embeddings stored in Google Drive.

This version avoids the previous local-copy step because copying thousands of small `.pt` files from Drive to `/content` was taking too long. The training therefore reads directly from:

`/content/drive/MyDrive/datasets/nsynth`

The notebook is configured for the `instrument_family` linear probe using `AFTER_Timbre` embeddings.


## 1. Mount Drive and configure paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys

REPO_ROOT = Path('/content/drive/MyDrive/SYNESIS/synesis')
DATASET_ROOT = Path('/content/drive/MyDrive/datasets/nsynth_reduced_5k')

assert REPO_ROOT.exists(), f'No encuentro el repo en {REPO_ROOT}'
assert DATASET_ROOT.exists(), f'No encuentro NSynth en {DATASET_ROOT}'

os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('Repo actual:', Path.cwd())
print('Dataset:', DATASET_ROOT)
print('config/features.py existe:', (REPO_ROOT / 'config' / 'features.py').exists())
print('synesis package existe:', (REPO_ROOT / 'synesis').exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo actual: /content/drive/MyDrive/SYNESIS/synesis
Dataset: /content/drive/MyDrive/datasets/nsynth_reduced_5k
config/features.py existe: True
synesis package existe: True


CREACIÓN DEL DATASET , NO HAY QUE VOLVER A EJECUTAR LA CELDA

In [ ]:
from collections import defaultdict
import json
import shutil
import random
from pathlib import Path

# =========================================================
# Create a larger reduced NSynth dataset from the existing one
# Around 5000 total examples
# =========================================================

SOURCE_ROOT = DATASET_ROOT  # Original dataset: /content/drive/MyDrive/datasets/nsynth
TARGET_ROOT = Path("/content/drive/MyDrive/datasets/nsynth_reduced_5k")

FEATURE = "AFTER_Timbre"
LABEL = "instrument_family"

# Around 5000 total examples across train/validation/test
N_TRAIN_PER_CLASS = 350  #350 ejemplos de cada classe
N_VAL_PER_CLASS = 75
N_TEST_PER_CLASS = 75

SEED = 42
random.seed(SEED)

CLASS_NAMES = [
    "bass",       # 0
    "brass",      # 1
    "flute",      # 2
    "guitar",     # 3
    "keyboard",   # 4
    "mallet",     # 5
    "organ",      # 6
    "reed",       # 7
    "string",     # 8
    "synth_lead", # 9
    "vocal",      # 10
]

assert SOURCE_ROOT.exists(), f"No existe el dataset original: {SOURCE_ROOT}"

print("Original dataset:", SOURCE_ROOT)
print("Reduced 5k dataset:", TARGET_ROOT)

def create_reduced_split(split, n_per_class):
    src_split = SOURCE_ROOT / split
    src_feature = src_split / FEATURE
    src_json = src_split / "examples.json"

    dst_split = TARGET_ROOT / split
    dst_feature = dst_split / FEATURE
    dst_json = dst_split / "examples.json"

    assert src_json.exists(), f"No existe: {src_json}"
    assert src_feature.exists(), f"No existe: {src_feature}"

    # This only removes the reduced split if it already exists.
    # It never touches the original dataset.
    if dst_split.exists():
        print(f"\nRemoving previous reduced split: {dst_split}")
        shutil.rmtree(dst_split)

    dst_feature.mkdir(parents=True, exist_ok=True)

    with open(src_json, "r") as f:
        metadata = json.load(f)

    by_class = defaultdict(list)

    for note_id, ex in metadata.items():
        label = int(ex[LABEL])
        pt_path = src_feature / f"{note_id}.pt"

        if pt_path.exists():
            by_class[label].append((note_id, ex))

    selected = {}

    print(f"\nCreating reduced split: {split}")

    for label_idx in range(len(CLASS_NAMES)):
        items = by_class[label_idx]
        random.shuffle(items)

        chosen = items[:n_per_class]

        print(
            f"{label_idx:2d} {CLASS_NAMES[label_idx]:12s}: "
            f"available={len(items):5d}, selected={len(chosen):4d}"
        )

        for note_id, ex in chosen:
            selected[note_id] = ex

            src_pt = src_feature / f"{note_id}.pt"
            dst_pt = dst_feature / f"{note_id}.pt"

            shutil.copy2(src_pt, dst_pt)

    with open(dst_json, "w") as f:
        json.dump(selected, f, indent=2)

    print(f"✅ {split}: saved {len(selected)} examples")
    print(f"   metadata: {dst_json}")
    print(f"   features: {dst_feature}")

# =========================================================
# Run creation
# =========================================================

create_reduced_split("train", N_TRAIN_PER_CLASS)
create_reduced_split("validation", N_VAL_PER_CLASS)
create_reduced_split("test", N_TEST_PER_CLASS)

print("\n✅ Reduced 5k dataset created successfully.")
print("Original dataset preserved at:", SOURCE_ROOT)
print("Reduced 5k dataset created at:", TARGET_ROOT)

Original dataset: /content/drive/MyDrive/datasets/nsynth
Reduced 5k dataset: /content/drive/MyDrive/datasets/nsynth_reduced_5k


COMPROBAR QUE EL DATASET ESTÁ BIEN CREADO

In [ ]:
from pathlib import Path
import json

TARGET_ROOT = Path("/content/drive/MyDrive/datasets/nsynth_reduced_5k")
FEATURE = "AFTER_Timbre"

assert TARGET_ROOT.exists(), f"No existe: {TARGET_ROOT}"

for split in ["train", "validation","test"]:
    json_path = TARGET_ROOT / split / "examples.json"
    feature_root = TARGET_ROOT / split / FEATURE

    assert json_path.exists(), f"No existe: {json_path}"
    assert feature_root.exists(), f"No existe: {feature_root}"

    with open(json_path, "r") as f:
        metadata = json.load(f)

    n_pt = len(list(feature_root.glob("*.pt")))

    print(f"{split}:")
    print("  examples.json:", len(metadata))
    print("  .pt files:", n_pt)

## 2. Check Python / NumPy environment

This environment previously worked with `numpy==1.26.4`. Run these cells before importing the Synesis modules.


In [3]:
!pip install -q --force-reinstall "numpy==1.26.4"
!pip install -q torchmetrics mir_eval wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 104.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 wh

In [2]:
import numpy as np
print("NumPy version:", np.__version__)

NumPy version: 1.26.4


## 3. Configure the evaluation task

For the current experiment, the notebook is configured for `instrument_family` classification.

A reasonable number of epochs for this setup is around **10**, but the actual number of epochs is controlled by the Synesis task configuration. If the framework still runs 100 epochs, stop it after the number of epochs you decide to keep.


In [3]:
FEATURE = 'AFTER_Timbre'
LABEL = 'instrument_family'
TASK = 'classification_SLP'

BATCH_SIZE_TRAIN = 64
SEED = 42
DESIRED_EPOCHS = 10

print('Feature:', FEATURE)
print('Label:', LABEL)
print('Task:', TASK)
print('Desired epochs:', DESIRED_EPOCHS)
print('Batch size train:', BATCH_SIZE_TRAIN)

Feature: AFTER_Timbre
Label: instrument_family
Task: classification_SLP
Desired epochs: 10
Batch size train: 64


## 4. Verify that embeddings exist in Drive

This checks that `examples.json` and the selected feature folder exist for each split.


In [4]:
for split in ['train', 'validation', 'test']:
    split_dir = DATASET_ROOT / split
    feature_dir = split_dir / FEATURE
    json_path = split_dir / 'examples.json'

    pt_count = len(list(feature_dir.glob('*.pt'))) if feature_dir.exists() else 0

    print(f'\nSplit: {split}')
    print('Split dir exists:', split_dir.exists())
    print('examples.json exists:', json_path.exists())
    print('Feature dir exists:', feature_dir.exists())
    print('Number of pt embeddings:', pt_count)

    assert split_dir.exists(), f'Missing split directory: {split_dir}'
    assert json_path.exists(), f'Missing examples.json: {json_path}'
    assert feature_dir.exists(), f'Missing feature directory: {feature_dir}'
    assert pt_count > 0, f'No .pt embeddings found in {feature_dir}'


Split: train
Split dir exists: True
examples.json exists: True
Feature dir exists: True
Number of pt embeddings: 3500

Split: validation
Split dir exists: True
examples.json exists: True
Feature dir exists: True
Number of pt embeddings: 750

Split: test
Split dir exists: True
examples.json exists: True
Feature dir exists: True
Number of pt embeddings: 750


## 5. Install the custom NSynth dataset inside Synesis

This cell writes `synesis/datasets/nsynth.py`.

Important details:
- The default dataset root points directly to Google Drive.
- Feature tensors are flattened with `reshape(-1)`, so `AFTER_Timbre` has shape `[6]` instead of `[1, 1, 6]`.
- The dataset supports both `instrument_family` and `qualities`, although this notebook is configured for `instrument_family`.


In [5]:
from pathlib import Path

nsynth_py = REPO_ROOT / "synesis" / "datasets" / "nsynth.py"

nsynth_py.write_text(r"""import json
from pathlib import Path
from typing import Optional, Tuple, Union

import numpy as np
import torch
from torch import Tensor
from torch.utils.data import Dataset

from config.features import configs as feature_configs
from synesis.datasets.dataset_utils import load_track


class FixedLabelEncoder:
    \"\"\"Small encoder object compatible with the Synesis downstream script.\"\"\"

    def __init__(self, classes):
        self.classes_ = np.array(classes)


class NSynth(Dataset):
    def __init__(
        self,
        feature: str,
        root: Union[str, Path] = "/content/drive/MyDrive/datasets/nsynth_reduced_5k",
        split: Optional[str] = None,
        download: bool = False,
        feature_config: Optional[dict] = None,
        audio_format: str = "wav",
        item_format: str = "feature",
        itemization: bool = True,
        seed: int = 42,
        label: str = "instrument_family",
        transform=None,
    ) -> None:
        self.tasks = ["instrument_family_classification", "qualities_classification"]
        self.fvs = ["instrument_family", "qualities"]

        if label not in self.fvs:
            raise ValueError(f"Invalid NSynth label: {label}. Options: {self.fvs}")

        if split not in [None, "train", "validation", "valid", "test"]:
            raise ValueError(
                f"Invalid split: {split}. Options: None, 'train', 'validation', 'valid', 'test'"
            )

        self.feature = feature
        self.root = Path(root)
        self.split = "validation" if split == "valid" else split
        self.label = label
        self.item_format = item_format
        self.itemization = itemization
        self.audio_format = audio_format

        if not feature_config:
            feature_config = feature_configs[feature]
        self.feature_config = feature_config

        if download:
            self._download()

        if self.label == "instrument_family":
            self.label_encoder = FixedLabelEncoder(list(range(11)))
        else:
            self.label_encoder = FixedLabelEncoder(
                [
                    "bright",
                    "dark",
                    "distortion",
                    "fast_decay",
                    "long_release",
                    "multiphonic",
                    "nonlinear_env",
                    "percussive",
                    "reverb",
                    "tempo-synced",
                ]
            )

        self._load_metadata()

    def _download(self) -> None:
        raise NotImplementedError(
            "NSynth download is not implemented here. Place the dataset under the expected root "
            "or pass root=/path/to/nsynth."
        )

    def _candidate_split_dirs(self):
        if self.split is None:
            return [
                self.root / "train",
                self.root / "validation",
                self.root / "test",
                self.root / "nsynth-train",
                self.root / "nsynth-valid",
                self.root / "nsynth-test",
            ]

        mapping = {
            "train": ["train", "nsynth-train", "nsynth_train"],
            "validation": ["validation", "valid", "nsynth-valid", "nsynth-validation", "nsynth_valid"],
            "test": ["test", "nsynth-test", "nsynth_test"],
        }
        return [self.root / name for name in mapping[self.split]]

    def _existing_split_dirs(self):
        split_dirs = [p for p in self._candidate_split_dirs() if (p / "examples.json").exists()]
        if not split_dirs:
            candidates = "\\n".join(str(p / "examples.json") for p in self._candidate_split_dirs())
            raise FileNotFoundError(
                "Could not find NSynth examples.json for the requested split. Tried:\\n"
                + candidates
            )
        return split_dirs

    def _audio_path_for_note(self, split_root: Path, note_id: str) -> Path:
        direct = split_root / f"{note_id}.{self.audio_format}"
        nested = split_root / "audio" / f"{note_id}.{self.audio_format}"
        if direct.exists():
            return direct
        return nested

    def _feature_path_for_note(self, split_root: Path, note_id: str) -> Path:
        return split_root / self.feature / f"{note_id}.pt"

    def _qualities_to_vector(self, qualities, note_id: str):
        if isinstance(qualities, dict):
            return [float(qualities[name]) for name in self.label_encoder.classes_]

        if len(qualities) != len(self.label_encoder.classes_):
            raise ValueError(
                f"Expected 10 NSynth qualities, got {len(qualities)} for note {note_id}"
            )

        return [float(q) for q in qualities]

    def _load_metadata(self) -> Tuple[list, torch.Tensor]:
        raw_data_paths = []
        feature_paths = []
        labels = []

        for split_root in self._existing_split_dirs():
            metadata_path = split_root / "examples.json"
            with open(metadata_path, "r") as f:
                metadata = json.load(f)

            for note_id, example in sorted(metadata.items()):
                raw_data_paths.append(self._audio_path_for_note(split_root, note_id))
                feature_paths.append(self._feature_path_for_note(split_root, note_id))

                if self.label == "instrument_family":
                    labels.append(int(example["instrument_family"]))
                elif self.label == "qualities":
                    labels.append(self._qualities_to_vector(example["qualities"], note_id))

        self.raw_data_paths = raw_data_paths
        self.feature_paths = feature_paths

        if self.label == "instrument_family":
            self.labels = torch.tensor(labels, dtype=torch.long)
        else:
            self.labels = torch.tensor(labels, dtype=torch.float)

        self.paths = self.raw_data_paths if self.item_format == "raw" else self.feature_paths

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> Tuple[Tensor, Tensor]:
        path = self.raw_data_paths[idx] if self.item_format == "raw" else self.feature_paths[idx]
        label = self.labels[idx]

        item_len_sec = self.feature_config.get("item_len_sec", None)

        track = load_track(
            path=path,
            item_format=self.item_format,
            itemization=self.itemization,
            item_len_sec=item_len_sec,
            sample_rate=self.feature_config["sample_rate"],
        )

        if self.item_format == "feature":
            track = track.reshape(-1)

        return track, label


Nsynth = NSynth
""")

print(f"✅ Wrote {nsynth_py}")


✅ Wrote /content/drive/MyDrive/SYNESIS/synesis/synesis/datasets/nsynth.py


## 6. Register NSynth in `dataset_utils.py`

This cell adds a small compatibility registration block only if it is not already present.


In [6]:
dataset_utils_path = REPO_ROOT / "synesis" / "datasets" / "dataset_utils.py"
text = dataset_utils_path.read_text()

registration_block = r"""
# ---- NSynth custom dataset registration ----
try:
    from synesis.datasets.nsynth import NSynth, Nsynth

    try:
        DatasetFactory.register_dataset("NSynth", NSynth)
    except Exception:
        pass

    for attr in ["datasets", "_datasets", "dataset_dict", "_dataset_dict"]:
        if hasattr(DatasetFactory, attr):
            try:
                getattr(DatasetFactory, attr)["NSynth"] = NSynth
                getattr(DatasetFactory, attr)["Nsynth"] = NSynth
            except Exception:
                pass
except Exception:
    pass
# ---- End NSynth custom dataset registration ----
"""

if "NSynth custom dataset registration" not in text:
    dataset_utils_path.write_text(text + "\n" + registration_block)
    print("✅ Registration block appended to dataset_utils.py")
else:
    print("✅ Registration block already present")

print("dataset_utils.py:", dataset_utils_path)

✅ Registration block already present
dataset_utils.py: /content/drive/MyDrive/SYNESIS/synesis/synesis/datasets/dataset_utils.py


## 7. Verify that NSynth loads the embeddings correctly

This is the most important check before training. The embedding should have shape `[6]`.


In [7]:
from pathlib import Path

nsynth_path = REPO_ROOT / "synesis" / "datasets" / "nsynth.py"

text = nsynth_path.read_text()

# Arreglar triple comillas escapadas que se han escrito mal en el archivo
text = text.replace('\\"""', '"""')
text = text.replace('""\\"', '"""')
text = text.replace('\\"', '"')

nsynth_path.write_text(text)

print("✅ nsynth.py corregido")
print("Archivo:", nsynth_path)

✅ nsynth.py corregido
Archivo: /content/drive/MyDrive/SYNESIS/synesis/synesis/datasets/nsynth.py


In [8]:
from config.features import configs as feature_configs
import importlib
import synesis.datasets.nsynth as nsynth_module

importlib.reload(nsynth_module)
NSynth = nsynth_module.NSynth

feature_config = {
    **feature_configs[FEATURE],
    "item_len_sec": 4,
    "sample_rate": 44100,
}

ds_train = NSynth(
    feature=FEATURE,
    split="train",
    item_format="feature",
    label=LABEL,
    feature_config=feature_config,
)

print("Root usado:", ds_train.root)
print("Number of train examples:", len(ds_train))

x, y = ds_train[0]
print("Embedding shape:", x.shape)
print("Label:", y)

assert tuple(x.shape) == (6,), f"Expected embedding shape [6], got {tuple(x.shape)}"
print("✅ Dataset check passed")

Root usado: /content/drive/MyDrive/datasets/nsynth_reduced_5k
Number of train examples: 3500
Embedding shape: torch.Size([6])
Label: tensor(0)
✅ Dataset check passed


# PASO PREVIO ENTRENARLO CON UN LINEAR PROBE NORMAL CON EL QUE NO SE ME ROMPE EL COLAB

In [ ]:
from pathlib import Path
import json
import torch
import numpy as np
import pandas as pd
import wandb

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

DATASET_ROOT = Path("/content/drive/MyDrive/datasets/nsynth")
FEATURE = "AFTER_Timbre"
LABEL = "instrument_family"

MAX_PER_CLASS_TRAIN = 300
MAX_PER_CLASS_VAL = 100
PRINT_EVERY = 250
SEED = 42

CLASS_NAMES = [
    "bass", "brass", "flute", "guitar", "keyboard",
    "mallet", "organ", "reed", "string", "synth_lead", "vocal"
]

rng = np.random.default_rng(SEED)

run = wandb.init(
    project="synesis",
    name="sklearn_logistic_regression_AFTER_Timbre_instrument_family_balanced",
    config={
        "feature": FEATURE,
        "label": LABEL,
        "model": "LogisticRegression",
        "classifier": "linear_probe_sklearn",
        "sampling": "balanced_by_class",
        "max_per_class_train": MAX_PER_CLASS_TRAIN,
        "max_per_class_val": MAX_PER_CLASS_VAL,
        "standard_scaler": True,
        "class_weight": "balanced",
        "max_iter": 5000,
        "seed": SEED,
    },
)

def select_balanced_items(metadata, max_per_class):
    by_class = {i: [] for i in range(len(CLASS_NAMES))}

    for note_id, ex in metadata.items():
        label = int(ex[LABEL])
        by_class[label].append((note_id, ex))

    selected = []

    print("\nAvailable examples per class:")
    for label_idx, items in by_class.items():
        rng.shuffle(items)
        chosen = items[:max_per_class]
        selected.extend(chosen)
        print(f"{label_idx:2d} {CLASS_NAMES[label_idx]:12s}: available={len(items):5d}, selected={len(chosen):4d}")

    rng.shuffle(selected)
    return selected

def load_split_balanced(split, max_per_class):
    split_root = DATASET_ROOT / split
    feature_root = split_root / FEATURE
    metadata_path = split_root / "examples.json"

    assert metadata_path.exists(), f"No existe metadata: {metadata_path}"
    assert feature_root.exists(), f"No existe feature folder: {feature_root}"

    with open(metadata_path, "r") as f:
        metadata = json.load(f)

    items = select_balanced_items(metadata, max_per_class=max_per_class)

    X = []
    y = []
    note_ids = []
    missing = 0

    print(f"\nLoading {split}: {len(items)} balanced examples...", flush=True)

    for i, (note_id, ex) in enumerate(items, start=1):
        pt_path = feature_root / f"{note_id}.pt"

        if not pt_path.exists():
            missing += 1
            continue

        emb = torch.load(pt_path, map_location="cpu")
        X.append(emb.reshape(-1).numpy())
        y.append(int(ex[LABEL]))
        note_ids.append(note_id)

        if i % PRINT_EVERY == 0:
            print(f"{split}: loaded {i}/{len(items)}", flush=True)
            wandb.log({f"loading/{split}_loaded": i})

    X = np.stack(X)
    y = np.array(y)

    print(f"✅ {split}: X={X.shape}, y={y.shape}, missing={missing}", flush=True)

    wandb.log({
        f"loading/{split}_final_loaded": len(X),
        f"loading/{split}_missing": missing,
    })

    return X, y, note_ids

X_train, y_train, train_ids = load_split_balanced("train", MAX_PER_CLASS_TRAIN)
X_val, y_val, val_ids = load_split_balanced("validation", MAX_PER_CLASS_VAL)

print("\nTrain label counts:")
print(pd.Series(y_train).map(lambda i: CLASS_NAMES[i]).value_counts().sort_index())

print("\nValidation label counts:")
print(pd.Series(y_val).map(lambda i: CLASS_NAMES[i]).value_counts().sort_index())

wandb.config.update({
    "num_train_examples": len(X_train),
    "num_val_examples": len(X_val),
    "embedding_dim": X_train.shape[1],
}, allow_val_change=True)

clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    ),
)

print("\nTraining LogisticRegression linear probe...", flush=True)
clf.fit(X_train, y_train)
print("✅ Training finished", flush=True)

y_pred = clf.predict(X_val)

acc = accuracy_score(y_val, y_pred)
macro_f1 = f1_score(y_val, y_pred, average="macro")
weighted_f1 = f1_score(y_val, y_pred, average="weighted")

print("\nValidation accuracy:", acc)
print("Validation macro F1:", macro_f1)
print("Validation weighted F1:", weighted_f1)

report_text = classification_report(
    y_val,
    y_pred,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
)

print("\nClassification report:")
print(report_text)

cm = confusion_matrix(y_val, y_pred, labels=list(range(len(CLASS_NAMES))))
print("\nConfusion matrix:")
print(cm)

wandb.log({
    "val/accuracy": acc,
    "val/macro_f1": macro_f1,
    "val/weighted_f1": weighted_f1,
})

report = classification_report(
    y_val,
    y_pred,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    output_dict=True,
    digits=4,
    zero_division=0,
)

report_df = pd.DataFrame(report).transpose()

wandb.log({
    "classification_report": wandb.Table(dataframe=report_df),
    "confusion_matrix": wandb.plot.confusion_matrix(
        preds=y_pred,
        y_true=y_val,
        class_names=CLASS_NAMES,
    )
})

output_dir = Path("/content/drive/MyDrive/SYNESIS/synesis/results_sklearn_probe")
output_dir.mkdir(parents=True, exist_ok=True)

report_csv = output_dir / "instrument_family_logistic_regression_balanced_report.csv"
cm_csv = output_dir / "instrument_family_logistic_regression_balanced_confusion_matrix.csv"

report_df.to_csv(report_csv)
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(cm_csv)

print("\nSaved report to:", report_csv)
print("Saved confusion matrix to:", cm_csv)

run.finish()
print("✅ W&B run finished")


Available examples per class:
 0 bass        : available= 2437, selected= 300
 1 brass       : available=  838, selected= 300
 2 flute       : available=  455, selected= 300
 3 guitar      : available= 1935, selected= 300
 4 keyboard    : available= 2221, selected= 300
 5 mallet      : available=  580, selected= 300
 6 organ       : available= 1474, selected= 300
 7 reed        : available=  654, selected= 300
 8 string      : available=  784, selected= 300
 9 synth_lead  : available=    0, selected=   0
10 vocal       : available=  363, selected= 300

Loading train: 3000 balanced examples...
train: loaded 250/3000
train: loaded 500/3000
train: loaded 750/3000
train: loaded 1000/3000
train: loaded 1250/3000
train: loaded 1500/3000
train: loaded 1750/3000
train: loaded 2000/3000
train: loaded 2250/3000
train: loaded 2500/3000
train: loaded 2750/3000
train: loaded 3000/3000
✅ train: X=(3000, 6), y=(3000,), missing=0

Available examples per class:
 0 bass        : available=  517, select

loading/train_final_loaded,▁
loading/train_loaded,▁▂▂▃▄▄▅▅▆▇▇█
loading/train_missing,▁
loading/validation_final_loaded,▁
loading/validation_loaded,▁▅█
loading/validation_missing,▁
val/accuracy,▁
val/macro_f1,▁
val/weighted_f1,▁
loading/train_final_loaded,3000
loading/train_loaded,3000


✅ W&B run finished


## 8. Check GPU availability

Do not launch the training if CUDA is not available.


In [9]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU: Tesla T4


## 9. Optional: connect to Weights & Biases

In [10]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nerea-gonzalez02 (nerea-gonzalez02-universitat-pompeu-fabra) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 10. Run informativeness / downstream probing

This command runs the linear probe with W&B logging enabled by default.

If the framework starts running 100 epochs and this is too long, stop the run manually after the number of epochs you decide to keep, for example after 5--10 completed epochs.


In [11]:
!python -m synesis.informativeness.downstream \
    -f $FEATURE \
    -d NSynth \
    -t $TASK \
    -l $LABEL

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: nerea-gonzalez02 (nerea-gonzalez02-universitat-pompeu-fabra) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in /content/drive/MyDrive/SYNESIS/synesis/wandb/run-20260605_091419-k2id3htj
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run INFO_DOWN_classification_SLP_NSynth_instrument_family_AFTER_Timbre
wandb: ⭐️ View project at https://wandb.ai/nerea-gonzalez02-universitat-pompeu-fabra/synesis
wandb: 🚀 View run at https://wandb.ai/nerea-gonzalez02-universitat-pompeu-fabra/synesis/runs/k2id3htj
Epoch 1/100: 100% 110/110 [1:05:04<00:00, 35.50s/it, loss=2.3916]
Epoch 1/100 - Validation: 100% 24/24 [13:31<00:00, 33.81s/it]
Computing Accuracy metric...
Computing F1 metric...
Epoch 1/100 -

In [ ]:
print("kernel available")

kernel available


## 11. Future `qualities` experiment

For `qualities`, the embeddings do not need to be re-extracted. Only the label and the task configuration change.


In [12]:
FEATURE = 'AFTER_Timbre'
LABEL = 'qualities'
TASK = 'tagging'

BATCH_SIZE_TRAIN = 64
SEED = 42
DESIRED_EPOCHS = 30

print('Feature:', FEATURE)
print('Label:', LABEL)
print('Task:', TASK)
print('Desired epochs:', DESIRED_EPOCHS)
print('Batch size train:', BATCH_SIZE_TRAIN)

Feature: AFTER_Timbre
Label: qualities
Task: tagging
Desired epochs: 30
Batch size train: 64


In [1]:
!python -m synesis.informativeness.downstream \
    -f $FEATURE \
    -d NSynth \
    -t $TASK \
    -l $LABEL

/usr/bin/python3: Error while finding module specification for 'synesis.informativeness.downstream' (ModuleNotFoundError: No module named 'synesis')
